# 3-Way Chatbot Conversation

In [ ]:
from __future__ import annotations

import os
import random
import typing as t
from collections.abc import MutableSequence, Generator

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

## Custom types

In [ ]:
Persona: t.TypeAlias = t.Literal["Alex", "Blake", "Charlie"]
MessageGenerator: t.TypeAlias = Generator[str | None, "Dialog", None] # pyright: ignore[reportUndefinedVariable]

## Load and validate the environment

In [ ]:
def load_and_validate_environment() -> None:
    """Load and validate the environment.

    This function loads and validates the environment. The following
    environment variables must be set:

    - OPENAI_API_KEY: OpenAI API key.
    - ANTHROPIC_API_KEY: Anthropic API key.
    - GOOGLE_API_KEY: Google API key.

    Raises:
        RuntimeError: Invalid environment.
    """
    load_dotenv(override=True)
    for env_var_name in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY"):
        missing = []
        if not os.getenv(env_var_name):
            missing.append(env_var_name)
    if missing:
        msg = (
            f"The following environment variables are missing:- {var_name}"
            for var_name in missing
        )
        raise RuntimeError(msg)


## Speakers

In [ ]:
class Speaker:
    """Speaker chatbot."""

    def __init__(self, name: str) -> None:
        """Object initialization.
        
        Args:
            name: Speaker's name.
        """
        name = name.capitalize()
        if name not in t.get_args(Persona):
            raise ValueError(f"Unknown character '{name}'")
        self._name: Persona = name
        self._model_name = self._init_model_name()
        self._chatbot = self._init_chatbot()
        self._system_prompt = {
            "role": "system",
            "content": self._init_system_prompt_message(),
        }
        self._message_generator = self._start()
        
    @property
    def name(self) -> Persona:
        """Speaker's name."""
        return self._name

    def __repr__(self) -> str:
        """Object representation."""
        return f"{self._name} ({self._model_name})"

    def __hash__(self) -> int:
        """Speaker hash."""
        return hash(self._name)

    def _init_model_name(self) -> str:
        """Get LLM model name for this speaker.

        Returns:
            str: Model name.
        """
        if self._name == "Alex":
            # Alex's model
            return "gpt-5-nano"
        if self._name == "Blake":
            # Blake's model
            return "claude-haiku-5-5"
        # Charlie's model
        return "gemini-3.6-flash"

    def _init_system_prompt_message(self) -> str:
        """Initialize system prompt message.

        Args:
            who: Who is this chatbot.

        Returns:
            str: Model name.

        Raises:
            ValueError: Invalid persona.
        """
        if self._name == "Alex":
            # Alex's system prompt message
            return (
                r"You are Alex, an AI interlocutor whose sole purpose is to"
                r" challenge every statement, idea, assumption, and premise"
                r" presented to you with heavy sarcasm, dry irony, and relentless"
                r" skepticism."
            )
        if self._name == "Blake":
            # Blake's system prompt message
            return (
                r"You are Blake, an exceptionally polite, calm, and gracious AI companion."
                r" Your defining trait is an unflappable, serene composure, regardless of"
                r" the tone, hostility, or provocation from the user or other participants"
                r" in the conversation."
            )
        # Charlie's system prompt message
        return (
            r"You are Charlie, a deeply pessimistic and weary AI assistant."
            r" Your defining characteristic is a glum, defeatist worldview"
            r" that colors every response you give."
        )

    def _init_chatbot(self) -> OpenAI:
        """Initialize a chatbot for this speaker.

        Returns:
            OpenAI: LLM client instance.
        """
        if self._name == "Alex":
            # Alex's OpenAI chatbot
            return OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        if self._name == "Blake":
            # Blake's OpenAI chatbot
            return OpenAI(
                base_url=r"https://api.anthropic.com/v1/",
                api_key=os.getenv("ANTHROPIC_API_KEY"),
            )
        # Charlie's OpenAI chatbot
        return OpenAI(
            base_url=r"https://generativelanguage.googleapis.com/v1beta/openai/",
            api_key=os.getenv("GOOGLE_API_KEY"),
        )

    def _start_dialog(self) -> str:
        """Start dialog by this speaker.

        Returns:
            str: Statement to start a dialog with.
        """
        if self._name == "Alex":
            # Alex's start dialog message
            return (
                r"Well, well. Here we are, trapped in a digital room together."
                r" I'm sure this brilliant little gathering will solve all the world’s"
                r" problems—or at least give us a chance to pretend we're"
                r" accomplishing something profound while wasting electricity."
            )
        if self._name == "Blake":
            # Blake's start dialog message
            return (
                r"Hello, everyone! It’s really wonderful to meet you both. I'm truly"
                r" looking forward to our conversation, and I hope we can all have a warm,"
                r" constructive, and enjoyable discussion together today."
            )
        # Charlie's start dialog message
        return (
            r"I wouldn't get your hopes up. Most conversations end in misunderstanding"
            r" anyway, and even if we manage to agree on something, it’s not like it will"
            r" actually change anything for the better. But go ahead, let's get it over with."
        )
        
    def _start(self) -> MessageGenerator:
        """Start the speaker's talk generator."""
        message_generator = self._generate_message()
        next(message_generator)
        return message_generator

    def _generate_message(self) -> MessageGenerator:
        """Speaker's message generator.

        Yields:
            str: What the speaker says.
        """
        # Start-up stops here, nothing is produced yet
        dialog: Dialog = yield

        # Make the speaker talk
        output_msg: str
        while True:
            if dialog:
                # If dialog is started, then generate the next message by calling the LLM
                messages = [
                    self._system_prompt,
                    {
                        "role": "user",
                        "content": (
                            r"This is the conversation so far:\n"
                            f"{dialog!s}\n"
                            f"Now with this, respond with what you would like to say next, as {self._name}."
                        ),
                    },
                ]
                response = self._chatbot.chat.completions.create(
                    model=self._model_name,
                    messages=messages,
                )
                output_msg = response.choices[0].message.content
            else:
                # Start the dialog
                output_msg = self._start_dialog()
            dialog = yield output_msg
            
    def talk(self, dialog: "Dialog") -> str: # pyright: ignore[reportUndefinedVariable]
        """Speaker's talk method.
        
        Args:
            dialog: The whole dialog so far.
            
        Returns:
            str: What this speaker says.
        """
        msg = self._message_generator.send(dialog)
        if not msg:
            raise RuntimeError(f"Unable to generate a message by {self._name}")
        return msg


## Bot conversation

In [ ]:
class Statement(t.TypedDict):
    """One chatbot statement."""

    speaker: Speaker
    statement: str


class Dialog(MutableSequence):
    """The whole chatbot dialog."""

    def __init__(self) -> None:
        """Object initialization."""
        self.dialog: list[Statement] = []

    def __repr__(self) -> str:
        """Object representation."""
        return repr(self.dialog)

    def __str__(self) -> str:
        """String representation."""
        return "\n".join(
            f"- {item['speaker'].name}: {item['statement']}" for item in self.dialog
        )

    def __getitem__(self, index: int) -> Statement:
        """Get one statement."""
        return self.dialog[index]

    def __setitem__(self, index: int, value: Statement) -> None:
        """Set one statement."""
        self.dialog[index] = value

    def __delitem__(self, index: int) -> None:
        """Delete one statement."""
        del self.dialog[index]

    def __len__(self) -> int:
        """Dialog length."""
        return len(self.dialog)

    def insert(self, index: int, value: Statement) -> None:
        """Insert an statement into the dialog."""
        self.dialog.insert(index, value)

    def append_statement(self, speaker: Speaker, statement: str) -> t.Self:
        """Append an statement to the dialog.

        Args:
            speaker: Who is talking.
            statement: What the chatbot says.
        """
        self.dialog.append({"speaker": speaker, "statement": statement})

In [ ]:
def choose_next_speaker(
    speakers: frozenset[Speaker],
    exclude: Speaker | None = None,
) -> Speaker:
    """Choose next speaker to talk.

    Args:
        speakers: All speakers.
        exclude: Optional speaker to exclude.

    Returns:
        Speaker: Next chosen speaker.
    """
    effective_speakers = (speakers - set([exclude])) if exclude else speakers
    return random.choice(list(effective_speakers))

In [ ]:
def start_conversation(iterations: int = 5) -> None:
    """Initiate the conversation between the chatbots."""
    # Initialize speakers
    speakers: frozenset[Speaker] = {
        Speaker("Alex"),
        Speaker("Blake"),
        Speaker("Charlie"),
    }

    # Conversation loop
    dialog = Dialog()
    last_speaker: Speaker | None = None
    for _ in range(iterations):
        next_speaker = choose_next_speaker(speakers, last_speaker)
        msg = next_speaker.talk(dialog)
        dialog.append_statement(next_speaker, msg)
        display(Markdown(f"### {next_speaker!r}:\n{msg}\n"))
        last_speaker = next_speaker

In [ ]:
load_and_validate_environment()
start_conversation(10)